<a href="https://colab.research.google.com/github/Abdallah387/medical_diagnosis_ai/blob/main/medical_diagnosis_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries

!pip install -U transformers datasets torch scikit-learn pandas accelerate sentencepiece

# Import libraries

import pandas as pd
import torch
from datasets import Dataset
from transformers import BertTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Load dataset

data = pd.read_csv("medical_data.csv")

# Map condition to numbers

unique_conditions = data['condition'].unique()
label_map = {condition: i for i, condition in enumerate(unique_conditions)}
data['labels'] = data['condition'].map(label_map)

num_labels = len(unique_conditions)

# Convert to HuggingFace dataset

dataset = Dataset.from_pandas(data)

# BioBERT model
model_name = "monologg/biobert_v1.1_pubmed"

tokenizer = BertTokenizer.from_pretrained(model_name) # Explicitly use BertTokenizer

# Tokenization

def tokenize(example):

  tokens = tokenizer(
  example["symptoms"],
  padding="max_length",
  truncation=True
  )
  tokens["labels"] = example["labels"]
  return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)

tokenized_dataset.set_format(
type="torch",
columns=["input_ids","attention_mask","labels"]
)

# Train test split

train_test = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = train_test["train"]
test_dataset = train_test["test"]

# Load BioBERT

model = AutoModelForSequenceClassification.from_pretrained(
model_name,
num_labels=num_labels
)

training_args = TrainingArguments(
output_dir="./results",
eval_strategy="epoch",
learning_rate=2e-5,
per_device_train_batch_size=8,
per_device_eval_batch_size=8,
num_train_epochs=3,
weight_decay=0.01
)

trainer = Trainer(
model=model,
args=training_args,
train_dataset=train_dataset,
eval_dataset=test_dataset
)

trainer.train()

# Save model

model.save_pretrained("medical_model")
tokenizer.save_pretrained("medical_model")

# Zip model

!zip -r medical_model.zip medical_model